# S03 · Demostración guiada: análisis tabular con Olist
### Python, NumPy, pandas y Git

**Diplomado:** Métodos estadísticos y matemáticos para la ciencia de datos · Facultad de Ciencias, UNAM  
**Módulo I:** Preprocesamiento y técnicas básicas de visualización de datos  
**Sesión 03 · 30 de octubre de 2026**

**Pregunta conductora:** *¿Cómo se distribuye la suma de precios registrados de artículos entre las categorías de productos de los pedidos entregados?*

En este recorrido estudiaremos la unidad de observación, la importación de tablas, filtros, uniones, agrupaciones y validaciones básicas. El código funciona con los archivos auténticos del repositorio público de Olist.

> **Interpretación:** `price` es el precio registrado de una fila de artículo. Su suma no representa automáticamente ingresos netos, pagos efectivamente recibidos ni facturación contable. `freight_value` se registra por separado.

## 0. Git y organización de carpetas

Desde una **terminal** de VS Code, sitúate en la carpeta en la que quieres guardar tu práctica y ejecuta:

```bash
git clone https://github.com/olist/work-at-olist-data.git
cd work-at-olist-data
git log -1 --oneline
git rev-parse --short HEAD
```

Esta práctica **clona un repositorio ajeno solo para leer sus datos**; no modificaremos sus archivos originales ni intentaremos publicar cambios en el repositorio de Olist.

Organización recomendada (dos carpetas hermanas):

```text
carpeta-del-curso/
├── work-at-olist-data/             ← repositorio clonado; incluye datasets/
└── S03_Olist_Material/
    ├── S03_Notebook_Demostracion_Python_v02.ipynb
    ├── S03_Olist_R_Quarto_v01.qmd
    └── requirements.txt
```

Si ya tienes el repositorio clonado, no hace falta clonarlo de nuevo. Para mostrar dónde estás, ejecuta `pwd` (macOS/Linux) o `cd` (PowerShell), y para conocer la versión del repositorio usa `git rev-parse HEAD` desde él.

**Si `git` falla o la red está restringida:** descarga el ZIP desde el mismo repositorio de GitHub y descomprímelo junto a esta carpeta. En ese caso habrá datos, pero no historial local de Git; para consultar su historial necesitarás un clonado con Git.

## 1. Entorno de ejecución

Utilizaremos **pandas**, **NumPy** y la biblioteca estándar. En la terminal del entorno de Python seleccionado en VS Code:

```bash
python -m pip install -r requirements.txt
```

En VS Code abre el notebook y selecciona su kernel de Python. Es posible instalar directamente `pandas`, `numpy`, `ipykernel` si no utilizas el archivo de dependencias.

Este notebook **no descarga automáticamente archivos**, para que quede visible y discutible de dónde proceden los datos.

In [1]:
from pathlib import Path
import os
import sys
import numpy as np
import pandas as pd
from IPython.display import display

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("NumPy:", np.__version__)

Python: 3.12.16
pandas: 2.3.3
NumPy: 2.5.4


In [2]:
! pip install pandas

## 2. Ubicar los CSV sin escribir rutas absolutas

Por defecto buscamos `work-at-olist-data/datasets` en la carpeta actual, en carpetas superiores y en carpetas hermanas. Si guardaste el repositorio en otro lugar, puedes establecer la variable de entorno `OLIST_DATA_DIR` con la ubicación de `datasets` antes de iniciar el kernel.

**No edites las rutas del repositorio original.**

In [3]:
REQUIRED_FILES = (
    "olist_orders_dataset.csv",
    "olist_order_items_dataset.csv",
    "olist_products_dataset.csv",
    "product_category_name_translation.csv",
)

def localizar_datos() -> Path:
    """Devuelve la carpeta datasets/ de Olist o indica cómo prepararla."""
    candidatos = []
    definida = os.environ.get("OLIST_DATA_DIR")
    if definida:
        candidatos.append(Path(definida).expanduser())

    ubicacion = Path.cwd().resolve()
    for base in (ubicacion, *ubicacion.parents):
        candidatos.extend([
            base / "work-at-olist-data" / "datasets",
            base / "datasets",
            base.parent / "work-at-olist-data" / "datasets",
        ])

    for ruta in candidatos:
        if ruta.is_dir() and all((ruta / nombre).is_file() for nombre in REQUIRED_FILES):
            return ruta
    raise FileNotFoundError(
        "No encontré los CSV de Olist. Ejecuta 'git clone "
        "https://github.com/olist/work-at-olist-data.git' en la "
        "carpeta del curso y coloca este notebook en una carpeta hermana "
        "del repositorio; o establece OLIST_DATA_DIR con la ruta a datasets/."
    )


DATA_DIR = localizar_datos()
print("Datos encontrados en:", DATA_DIR)

Datos encontrados en: /Users/davidmoreno/Documents/UNAM/Diplomados/DMEMCDM/Diplomado_2026/DMEMCD_2026/work-at-olist-data/datasets


## 3. Importar datos y examinar su estructura

**Unidad de observación (muy importante):**

- `orders`: un pedido (`order_id`).
- `order_items`: una fila de artículo dentro de un pedido (`order_id`, `order_item_id`).
- `products`: un producto (`product_id`).
- `category_translation`: una equivalencia de nombre de categoría.

**Advertencia:** una fila de `order_items` no es necesariamente un pedido completo ni una unidad física agregada para todos los productos.

In [13]:
orders = pd.read_csv(
    DATA_DIR / "olist_orders_dataset.csv",
    parse_dates=["order_purchase_timestamp", "order_delivered_customer_date"],
)
items = pd.read_csv(DATA_DIR / "olist_order_items_dataset.csv")
products = pd.read_csv(DATA_DIR / "olist_products_dataset.csv")
categories = pd.read_csv(DATA_DIR / "product_category_name_translation.csv")

for nombre, tabla in {
    "orders": orders,
    "items": items,
    "products": products,
    "categories": categories,
}.items():
    print(f"{nombre:12s}: {len(tabla):>8,} filas × {tabla.shape[1]:>2} columnas")

display(orders.head(3))
display(items.head(3))
display(products.head(3))
display(categories.head(3))

orders      :   99,441 filas ×  8 columnas
items       :  112,650 filas ×  7 columnas
products    :   32,951 filas ×  9 columnas
categories  :       71 filas ×  2 columnas


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.9,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.9,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.0,17.87


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto


In [12]:
orders2.loc[0,"order_purchase_timestamp"]

'2017-10-02 10:56:33'

### 3.1. Preguntas para interpretar las tablas

1. ¿Cómo identificarías un **pedido**? ¿Y una **fila de artículo de un pedido**?
2. ¿Qué pasa si contamos filas en `items` como si fueran pedidos?
3. ¿Por qué las columnas `order_id` y `product_id` deben conservarse como identificadores, no como magnitudes numéricas?

Antes de analizar valores, identifica los tipos y algunos valores faltantes.

In [ ]:
print("Tipos en orders:")
print(orders.dtypes.to_string())
print("\nFaltantes en products (primeras columnas con ausencias):")
display(products.isna().sum().sort_values(ascending=False).head(8).to_frame("n_faltantes"))

Tipos en orders:
order_id                                 object
customer_id                              object
order_status                             object
order_purchase_timestamp         datetime64[ns]
order_approved_at                        object
order_delivered_carrier_date             object
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date            object

Faltantes en products (primeras columnas con ausencias):


,n_faltantes
product_category_name,610
product_name_lenght,610
product_description_lenght,610
product_photos_qty,610
product_weight_g,2
product_length_cm,2
product_height_cm,2
product_width_cm,2


## 4. Primeras operaciones: seleccionar, filtrar y transformar

Una pregunta concreta: ¿cuántas filas de artículos corresponden a **pedidos entregados**?

Una selección puede describirse como el subconjunto de filas que satisface un predicado: \(D_P=\{x\in D:P(x)\}\). Primero necesitamos incorporar el estado del pedido a cada artículo.

In [18]:
# Validaciones previas: son restricciones esperadas del modelo relacional.
print("order_id repetidos en orders:", orders["order_id"].duplicated().sum())
print("(order_id, order_item_id) repetidos en items:",
      items.duplicated(["order_id", "order_item_id"]).sum())
print("product_id repetidos en products:", products["product_id"].duplicated().sum())

assert orders["order_id"].is_unique, "La llave order_id no es única en orders."
assert products["product_id"].is_unique, "La llave product_id no es única en products."
assert not items.duplicated(["order_id", "order_item_id"]).any(), (
    "Se repite la llave compuesta de artículos."
)

order_id repetidos en orders: 0
(order_id, order_item_id) repetidos en items: 0
product_id repetidos en products: 0


In [19]:
# Relación muchos artículos → un pedido. validate evita una expansión inesperada.
items_orders = items.merge(
    orders[["order_id", "order_status", "order_purchase_timestamp"]],
    on="order_id",
    how="left",
    validate="many_to_one",
    indicator=True,
)

print("Correspondencia con pedidos:")
display(items_orders["_merge"].value_counts(dropna=False).to_frame("filas"))
print("¿Se mantuvo el número de filas de artículos?", len(items_orders) == len(items))

# No borramos filas sin correspondencia silenciosamente: las conservamos para auditoría.
items_orders = items_orders.drop(columns="_merge")
items_entregados = items_orders.loc[
    items_orders["order_status"].eq("delivered")
].copy()

print("Filas de artículos en pedidos con estado delivered:", len(items_entregados))
display(items_entregados[["order_id", "product_id", "price", "freight_value"]].head())

Correspondencia con pedidos:


,filas
_merge,
both,112650
left_only,0
right_only,0


¿Se mantuvo el número de filas de artículos? True
Filas de artículos en pedidos con estado delivered: 110197


,order_id,product_id,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,4244733e06e7ecb4970a6e2683c13e61,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,e5f2d52b802189ee658865ca93d83a8f,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,c777355d18b72b67abbeef9df44fd0fd,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,7634da152a4610f1595efa32f14722fc,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,ac6c3623068f30de03045865e4e10089,199.90,18.14


**Nota de interpretación.** El estado `delivered` es un criterio explícito del análisis, no una garantía de que los pagos hayan sido cobrados o de que no existan devoluciones. Tampoco debemos sumar `freight_value` al precio sin explicar qué concepto económico pretendemos representar.

## 5. Unir productos y nombres de categorías

En una relación **muchos a uno**, múltiples filas de artículos se vinculan con el mismo `product_id` del catálogo. Si la llave del catálogo estuviera duplicada, una unión podría multiplicar filas. Por eso comprobamos unicidad y `validate='many_to_one'`.

In [20]:
assert categories["product_category_name"].is_unique, (
    "El diccionario de categorías contiene llaves repetidas."
)

analisis = items_entregados.merge(
    products[["product_id", "product_category_name", "product_weight_g"]],
    on="product_id", how="left", validate="many_to_one", indicator="_origen_producto",
)
analisis = analisis.merge(
    categories,
    on="product_category_name", how="left", validate="many_to_one",
    indicator="_origen_traduccion",
)

print("Estados de correspondencia de producto:")
display(analisis["_origen_producto"].value_counts(dropna=False))
print("Estados de correspondencia de traducción:")
display(analisis["_origen_traduccion"].value_counts(dropna=False))

# Respetar y señalar los valores ausentes; el nombre de categoría no siempre existe.
analisis["categoria"] = (
    analisis["product_category_name_english"]
    .fillna(analisis["product_category_name"])
    .fillna("SIN_CATEGORIA")
)
assert len(analisis) == len(items_entregados), (
    "La unión alteró inesperadamente el número de filas."
)

display(analisis[["order_id", "product_id", "categoria", "price"]].head())

Estados de correspondencia de producto:


_origen_producto
both          110197
left_only          0
right_only         0
Name: count, dtype: int64

Estados de correspondencia de traducción:


_origen_traduccion
both          108638
left_only       1559
right_only         0
Name: count, dtype: int64

,order_id,product_id,categoria,price
0,00010242fe8c5a6d1ba2dd792cb16214,4244733e06e7ecb4970a6e2683c13e61,cool_stuff,58.90
1,00018f77f2f0320c557190d7a144bdd3,e5f2d52b802189ee658865ca93d83a8f,pet_shop,239.90
2,000229ec398224ef6ca0657da4fc703e,c777355d18b72b67abbeef9df44fd0fd,furniture_decor,199.00
3,00024acbcdf0a6daa1e931b038114c75,7634da152a4610f1595efa32f14722fc,perfumery,12.99
4,00042b26cf59d7ce69dfabb4e55b4fd9,ac6c3623068f30de03045865e4e10089,garden_tools,199.90


In [34]:
analisis.categoria.isna().sum()

np.int64(0)

## 6. Agrupar, resumir e interpretar

Si $p_i$ denota el precio registrado de la fila de artículo $i$ y $g(i)$ su categoría, calculamos:

$$S_c = \sum_{i\, :\, g(i)=c} p_i .$$

Esto mide la **suma de precios registrados de artículos** por categoría dentro de la muestra y filtro elegidos. No es una medida validada de ingresos contables. Separadamente calculamos número de filas de artículos, número de pedidos distintos y mediana del precio.

In [35]:
resumen_categorias = (
    analisis
    .groupby("categoria", dropna=False)
    .agg(
        filas_articulos=("order_item_id", "size"),
        pedidos_distintos=("order_id", "nunique"),
        suma_precios_articulos=("price", "sum"),
        mediana_precio_articulo=("price", "median"),
    )
    .sort_values("suma_precios_articulos", ascending=False)
    .reset_index()
)

display(resumen_categorias.head(10))
print("Suma en la tabla base:", analisis["price"].sum())
print("Suma de los resúmenes:", resumen_categorias["suma_precios_articulos"].sum())
assert np.isclose(
    analisis["price"].sum(), resumen_categorias["suma_precios_articulos"].sum()
), "Los totales no coinciden: revisa el proceso de agrupación."

,categoria,filas_articulos,pedidos_distintos,suma_precios_articulos,mediana_precio_articulo
0,health_beauty,9465,8647,1233131.72,79.90
1,watches_gifts,5859,5495,1166176.98,128.99
2,bed_bath_table,10953,9272,1023434.76,79.05
3,sports_leisure,8431,7530,954852.55,77.90
4,computers_accessories,7644,6530,888724.61,81.99
5,furniture_decor,8160,6307,711927.69,65.49
6,housewares,6795,5743,615628.69,59.70
7,cool_stuff,3718,3559,610204.10,129.90
8,auto,4140,3810,578966.65,84.90
9,toys,4030,3804,471286.48,79.90


Suma en la tabla base: 13221498.11
Suma de los resúmenes: 13221498.11


### 6.1. Crear variables y introducir NumPy

Este umbral de **100 unidades monetarias** es solo un criterio didáctico para practicar una transformación. No es una segmentación validada de Olist.

In [36]:
analisis["tramo_precio_demo"] = np.where(
    analisis["price"].ge(100), "100 o más", "Menos de 100"
)

display(
    analisis.groupby("tramo_precio_demo", dropna=False)
    .agg(filas_articulos=("price", "size"), mediana_precio=("price", "median"))
)

,filas_articulos,mediana_precio
tramo_precio_demo,,
100 o más,39461,164.0
Menos de 100,70736,49.5


## 7. Funciones, condicionales y ciclos: aplicar programación a los datos

Una función concentra una operación repetible con parámetros explícitos. Un condicional controla casos válidos. Un ciclo es útil en algunas tareas repetitivas, aunque para agrupar grandes tablas suele ser preferible la operación vectorizada de pandas.

In [37]:
def resumen_por_estado(tabla: pd.DataFrame, estado: str = "delivered") -> pd.DataFrame:
    """Resume precios por estado, sin alterar la tabla recibida."""
    if estado not in tabla["order_status"].dropna().unique():
        raise ValueError(f"Estado no encontrado en los datos: {estado!r}")
    seleccion = tabla.loc[tabla["order_status"].eq(estado)]
    return (
        seleccion.groupby("order_status", dropna=False)
        .agg(filas_articulos=("price", "size"),
             suma_precios_articulos=("price", "sum"))
        .reset_index()
    )

print("Ejemplo con función:")
display(resumen_por_estado(items_orders, "delivered"))

print("Ejemplo con ciclo (no sustituye las agrupaciones de pandas):")
for estado in ["delivered", "canceled"]:
    n = int(items_orders["order_status"].eq(estado).sum())
    print(f"  {estado}: {n} filas de artículos")

Ejemplo con función:


,order_status,filas_articulos,suma_precios_articulos
0,delivered,110197,13221498.11


Ejemplo con ciclo (no sustituye las agrupaciones de pandas):
  delivered: 110197 filas de artículos
  canceled: 542 filas de artículos


## 8. Verificar antes de interpretar

Nuestro resumen depende de varias decisiones: filtrar pedidos con estado `delivered`, relacionar artículos con sus catálogos y conservar las categorías no identificadas como `SIN_CATEGORIA`.

Contesta antes de continuar:

1. ¿Qué representa una fila de `items` y qué representa una fila de `orders`?
2. ¿Por qué el número de pedidos distintos puede ser menor que el de filas de artículos?
3. ¿En qué condiciones una unión multiplicaría filas y modificaría indebidamente la suma de precios?
4. ¿Qué información falta para equiparar nuestros totales con ingresos efectivos?

In [38]:
# Comprobaciones finales sobre el conjunto filtrado.
print("Filas tras filtrar y unir:", len(analisis))
print("Pedidos distintos:", analisis["order_id"].nunique())
print("Artículos sin categoría identificable:", analisis["categoria"].eq("SIN_CATEGORIA").sum())

assert len(analisis) == len(items_entregados)
assert not analisis["order_id"].isna().any()
assert np.isclose(
    analisis["price"].sum(),
    resumen_categorias["suma_precios_articulos"].sum(),
)

Filas tras filtrar y unir: 110197
Pedidos distintos: 96478
Artículos sin categoría identificable: 1537


## 9. Reproducibilidad y continuidad

Para reproducir el análisis hacen falta la **fuente de datos**, una referencia a su versión, el **código**, las **dependencias** y la documentación de las decisiones sobre filtros y uniones.

Desde la terminal, dentro del repositorio clonado, registra el identificador completo del commit:

```bash
git rev-parse HEAD
```

En el laboratorio pondremos a prueba el mismo modelo relacional. En S04 estudiaremos las distribuciones y medidas descriptivas que permiten interpretar mejor estas cantidades.

## Referencias y límites de comprobación

- Olist. *Work at Olist Data*, repositorio público: https://github.com/olist/work-at-olist-data (fuente primaria de los CSV).
- pandas. *User Guide*: https://pandas.pydata.org/docs/user_guide/
- NumPy. *Documentation*: https://numpy.org/doc/stable/
- Git. *Pro Git*: https://git-scm.com/book/es/v2

**Lectura de resultados:** los conteos, totales y distribuciones dependen de la versión real clonada. El notebook comprueba llaves y cardinalidades al ejecutarse, en lugar de dar por verificadas cantidades que no se han observado.

**Ética y alcance:** no atribuir a las cifras carácter de ventas totales de Olist ni de pagos recibidos; no difundir información personal de los archivos de otras tablas sin evaluar su uso y condiciones de acceso.